In [1]:
import warnings
warnings.filterwarnings("ignore")
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MaxAbsScaler, label_binarize
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score
)

In [2]:
current_dir = Path.cwd()
BASE_DIR = current_dir.parent if current_dir.name in ["src", "notebooks"] else current_dir
 
REAL_CSV      = BASE_DIR / "datasets" / "processed" / "humaid_processed.csv"
AUGMENTED_CSV = BASE_DIR / "datasets" / "processed" / "humaid_train_augmented.csv"
 
# All trained ML artefacts live here, split by augmentation suffix
TRAINED_MODELS_DIR = BASE_DIR / "trained_models" / "ml_baselines"
 
# Training results (plots, CSVs) go under results/Training/
RESULTS_DIR = BASE_DIR / "results" / "Training"
 
for d in [TRAINED_MODELS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [3]:
LABEL_NAMES = [
    "Critical Rescue",           # 0
    "Resource Requests",         # 1
    "Situational Awareness",     # 2
    "Volunteering and Donations",# 3
    "Irrelevant",                # 4
]
label2id = {label: i for i, label in enumerate(LABEL_NAMES)}
id2label  = {i: label for label, i in label2id.items()}
 
CLASS_COLORS = ["#d32f2f", "#ff9800", "#1976d2", "#4caf50", "#9e9e9e"]
RANDOM_SEED  = 42

In [4]:
def load_splits(use_augmentation: bool):
    real_df  = pd.read_csv(REAL_CSV)
    dev_df   = real_df[real_df["split"] == "dev"].copy()
    test_df  = real_df[real_df["split"] == "test"].copy()
    train_df = (pd.read_csv(AUGMENTED_CSV)
                if use_augmentation
                else real_df[real_df["split"] == "train"].copy())
 
    for df in [train_df, dev_df, test_df]:
        df["clean_text"] = df["clean_text"].astype(str).fillna("")
        df["label"]      = df["target_label"].map(label2id).astype(int)
    return train_df, dev_df, test_df

In [5]:
def vectorize(train_texts, other_text_sets):
    """
    Fits TF-IDF + MaxAbsScaler on train_texts.
    Returns (vectorizer, scaler, X_train_scaled, [X_other_scaled, ...]).
    Both artefacts must be saved so the Evaluation notebook can replicate
    the exact same transformation at inference time.
    """
    vec    = TfidfVectorizer(
        max_features=30000, ngram_range=(1, 2),
        min_df=3, max_df=0.95, sublinear_tf=True
    )
    scaler = MaxAbsScaler()
 
    X_train_raw    = vec.fit_transform(train_texts)
    X_train_scaled = scaler.fit_transform(X_train_raw)
    others_scaled  = [scaler.transform(vec.transform(t)) for t in other_text_sets]
 
    return vec, scaler, X_train_scaled, others_scaled

In [6]:
def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    macro_p,  macro_r,  macro_f1,  _ = precision_recall_fscore_support(y_true, y_pred, average="macro",    zero_division=0)
    weight_p, weight_r, weight_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    class_p,  class_r,  class_f1,  _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0, labels=list(range(len(LABEL_NAMES)))
    )
 
    metrics = {
        "accuracy":           acc,
        "macro_precision":    macro_p,  "macro_recall":    macro_r,  "macro_f1":    macro_f1,
        "weighted_precision": weight_p, "weighted_recall": weight_r, "weighted_f1": weight_f1,
    }
    for i, name in enumerate(LABEL_NAMES):
        safe = name.replace(" ", "_").lower()
        metrics[f"precision_class_{safe}"] = float(class_p[i])
        metrics[f"recall_class_{safe}"]    = float(class_r[i])
        metrics[f"f1_class_{safe}"]        = float(class_f1[i])
    return metrics

In [7]:
def plot_confusion_matrix(y_true, y_pred, title, out_path):
    cm      = confusion_matrix(y_true, y_pred, labels=list(range(len(LABEL_NAMES))))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, data, fmt, subtitle in zip(
        axes, [cm, cm_norm], ["d", ".2f"], ["Counts", "Row-Normalized"]
    ):
        sns.heatmap(
            data, annot=True, fmt=fmt, cmap="Blues",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            ax=ax, cbar=True, vmin=0, vmax=None if fmt == "d" else 1
        )
        ax.set_title(f"{title}\n({subtitle})")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.tick_params(axis="x", rotation=35)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()

In [8]:
def plot_roc_pr_curves(y_true, y_proba, title, out_dir, prefix):
    y_bin  = label_binarize(y_true, classes=list(range(len(LABEL_NAMES))))
    fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
    fig_pr,  ax_pr  = plt.subplots(figsize=(8, 6))
    for i, cls in enumerate(LABEL_NAMES):
        fpr,  tpr,  _ = roc_curve(y_bin[:, i], y_proba[:, i])
        prec, rec,  _ = precision_recall_curve(y_bin[:, i], y_proba[:, i])
        ax_roc.plot(fpr, tpr,  color=CLASS_COLORS[i], lw=2,
                    label=f"{cls} (AUC={auc(fpr, tpr):.3f})")
        ax_pr.plot(rec,  prec, color=CLASS_COLORS[i], lw=2,
                   label=f"{cls} (AP={average_precision_score(y_bin[:, i], y_proba[:, i]):.3f})")
    for ax, t, xl, yl in zip(
        [ax_roc, ax_pr], ["ROC", "PR"], ["FPR", "Recall"], ["TPR", "Precision"]
    ):
        if t == "ROC":
            ax.plot([0, 1], [0, 1], "k--", lw=1)
        ax.set_title(f"{title} - {t}")
        ax.set_xlabel(xl)
        ax.set_ylabel(yl)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    fig_roc.savefig(out_dir / f"{prefix}_roc.png", dpi=150, bbox_inches="tight")
    fig_pr.savefig( out_dir / f"{prefix}_pr.png",  dpi=150, bbox_inches="tight")
    plt.close(fig_roc)
    plt.close(fig_pr)

In [9]:
def plot_ablation(df, out_dir):
    for metric, title in [
        ("test_macro_f1",                    "Macro F1"),
        ("test_f1_class_resource_requests",  "Resource Requests F1"),
    ]:
        pivot = df.pivot(index="model_name", columns="augmentation_used", values=metric)
        fig, ax = plt.subplots(figsize=(10, 6))
        pivot.plot(kind="bar", ax=ax, color=["#1976d2", "#ff9800"], edgecolor="white")
        ax.set_title(f"{title}: Baseline vs Augmented (Test Set)")
        ax.set_ylabel(title)
        ax.tick_params(axis="x", rotation=0)
        ax.legend(["No Augmentation", "Augmented"])
        ax.grid(axis="y", alpha=0.3)
        ax.set_ylim(0, 1)
        for container in ax.containers:
            ax.bar_label(container, fmt="%.3f")
        plt.tight_layout()
        plt.savefig(out_dir / f"ablation_{metric}.png", dpi=150, bbox_inches="tight")
        plt.close()

In [10]:
def build_models():
    return {
        "Logistic_Regression": LogisticRegression(
            class_weight="balanced", max_iter=1000,
            random_state=RANDOM_SEED, n_jobs=-1
        ),
        "Linear_SVM": LinearSVC(
            class_weight="balanced", C=1.0,
            max_iter=5000, random_state=RANDOM_SEED
        ),
        "Naive_Bayes": MultinomialNB(alpha=0.1),
    }

In [11]:
def run_ml_experiment(use_augmentation: bool):
    train_df, dev_df, test_df = load_splits(use_augmentation)
    suffix = "aug" if use_augmentation else "no_aug"
 
    # ── Output directories ────────────────────────────────────────────
    # Models: trained_models/ml_baselines/{suffix}/
    model_dir = TRAINED_MODELS_DIR / suffix
    model_dir.mkdir(parents=True, exist_ok=True)
 
    # Results: results/Training/ml_baselines_{suffix}/
    run_dir = RESULTS_DIR / f"ml_baselines_{suffix}"
    run_dir.mkdir(parents=True, exist_ok=True)
 
    # ── Vectorise + scale ─────────────────────────────────────────────
    vec, scaler, X_train, (X_dev, X_test) = vectorize(
        train_df["clean_text"],
        [dev_df["clean_text"], test_df["clean_text"]]
    )
    y_train = train_df["label"].values
    y_dev   = dev_df["label"].values
    y_test  = test_df["label"].values
 
    sample_weights = compute_sample_weight("balanced", y=y_train)
 
    # Save vectorizer AND scaler — Evaluation notebook needs both
    joblib.dump(vec,    model_dir / "tfidf_vectorizer.joblib")
    joblib.dump(scaler, model_dir / "tfidf_scaler.joblib")
    print(f"[{suffix}] Saved vectorizer + scaler → {model_dir}")
 
    # ── Train & evaluate each model ───────────────────────────────────
    models  = build_models()
    results = []
 
    for name, model in models.items():
        print(f"\n[{suffix}] Training {name}...")
 
        if isinstance(model, LinearSVC):
            model = CalibratedClassifierCV(model, cv=3, method="sigmoid")
            model.fit(X_train, y_train, sample_weight=sample_weights)
        elif name == "Naive_Bayes":
            model.fit(X_train, y_train, sample_weight=sample_weights)
        else:
            model.fit(X_train, y_train)
 
        # Save model weights
        # Path: trained_models/ml_baselines/{suffix}/{ModelName}.joblib
        joblib.dump(model, model_dir / f"{name}.joblib")
 
        dev_pred  = model.predict(X_dev)
        test_pred = model.predict(X_test)
        test_proba = model.predict_proba(X_test)
 
        dev_m  = compute_metrics(y_dev,  dev_pred)
        test_m = compute_metrics(y_test, test_pred)
 
        # Save plots into results/Training/ml_baselines_{suffix}/
        plot_confusion_matrix(y_dev,  dev_pred,
                              f"{name} ({suffix}) - Dev",
                              run_dir / f"{name}_cm_dev.png")
        plot_confusion_matrix(y_test, test_pred,
                              f"{name} ({suffix}) - Test",
                              run_dir / f"{name}_cm_test.png")
        plot_roc_pr_curves(y_test, test_proba, f"{name} ({suffix})", run_dir, name)
 
        row = {"model_name": name, "augmentation_used": use_augmentation}
        for k, v in dev_m.items():  row[f"dev_{k}"]  = v
        for k, v in test_m.items(): row[f"test_{k}"] = v
        results.append(row)
        print(f"  dev_macro_f1={dev_m['macro_f1']:.4f}  test_macro_f1={test_m['macro_f1']:.4f}")
 
    return pd.DataFrame(results)

In [12]:
results_no_aug = run_ml_experiment(use_augmentation=False)
results_aug    = run_ml_experiment(use_augmentation=True)
final_df       = pd.concat([results_no_aug, results_aug], ignore_index=True)


[no_aug] Saved vectorizer + scaler → /home/aakash/rizwan/NLP_Project/trained_models/ml_baselines/no_aug

[no_aug] Training Logistic_Regression...
  dev_macro_f1=0.7530  test_macro_f1=0.7480

[no_aug] Training Linear_SVM...
  dev_macro_f1=0.7196  test_macro_f1=0.7185

[no_aug] Training Naive_Bayes...
  dev_macro_f1=0.6919  test_macro_f1=0.6905
[aug] Saved vectorizer + scaler → /home/aakash/rizwan/NLP_Project/trained_models/ml_baselines/aug

[aug] Training Logistic_Regression...
  dev_macro_f1=0.7485  test_macro_f1=0.7493

[aug] Training Linear_SVM...
  dev_macro_f1=0.7429  test_macro_f1=0.7383

[aug] Training Naive_Bayes...
  dev_macro_f1=0.6883  test_macro_f1=0.6880


In [13]:
combined_csv = RESULTS_DIR / "ml_baselines_all_results.csv"
final_df.to_csv(combined_csv, index=False)
print(f"\nCombined results saved → {combined_csv}")
 
# Ablation plots (both runs together)
ablation_dir = RESULTS_DIR / "ablation"
ablation_dir.mkdir(exist_ok=True)
plot_ablation(final_df, ablation_dir)


Combined results saved → /home/aakash/rizwan/NLP_Project/results/Training/ml_baselines_all_results.csv


In [14]:
dev_summary_columns = [
    "model_name", "augmentation_used",
    "dev_accuracy", "dev_macro_precision",
    "dev_precision_class_resource_requests",
]
 
print("=" * 100)
print(f"{'ML BASELINES: DEV SET PERFORMANCE COMPARISON':^100}")
print("=" * 100)

dev_comparison_table = final_df[dev_summary_columns].sort_values(
    by=["model_name", "augmentation_used"]
)
print(dev_comparison_table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("=" * 100)
 
for model in dev_comparison_table["model_name"].unique():
    m_df     = dev_comparison_table[dev_comparison_table["model_name"] == model]
    no_aug   = m_df[m_df["augmentation_used"] == False]["dev_precision_class_resource_requests"].values[0]
    aug      = m_df[m_df["augmentation_used"] == True ]["dev_precision_class_resource_requests"].values[0]
    print(f"{model.replace('_', ' '):<20} | Resource Req. Dev F1 Delta: {aug - no_aug:+.4f}")

                            ML BASELINES: DEV SET PERFORMANCE COMPARISON                            
         model_name  augmentation_used  dev_accuracy  dev_macro_precision  dev_precision_class_resource_requests
         Linear_SVM              False        0.7664               0.7062                                 0.3298
         Linear_SVM               True        0.7812               0.7317                                 0.4742
Logistic_Regression              False        0.7916               0.7464                                 0.5033
Logistic_Regression               True        0.7907               0.7467                                 0.5148
        Naive_Bayes              False        0.7429               0.6824                                 0.3890
        Naive_Bayes               True        0.7470               0.7015                                 0.5081
Linear SVM           | Resource Req. Dev F1 Delta: +0.1444
Logistic Regression  | Resource Req. Dev F1 Delta

In [15]:
dev_macro_columns = [
    "model_name", "augmentation_used",
    "dev_accuracy", "dev_macro_precision", "dev_macro_recall", "dev_macro_f1",
]
 
print("=" * 108)
print(f"{'ML BASELINES: DEV SET MACRO METRICS COMPARISON':^108}")
print("=" * 108)
 
dev_macro_table = final_df[dev_macro_columns].sort_values(
    by=["model_name", "augmentation_used"]
)
print(dev_macro_table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("=" * 108)

                               ML BASELINES: DEV SET MACRO METRICS COMPARISON                               
         model_name  augmentation_used  dev_accuracy  dev_macro_precision  dev_macro_recall  dev_macro_f1
         Linear_SVM              False        0.7664               0.7062            0.7627        0.7196
         Linear_SVM               True        0.7812               0.7317            0.7573        0.7429
Logistic_Regression              False        0.7916               0.7464            0.7610        0.7530
Logistic_Regression               True        0.7907               0.7467            0.7505        0.7485
        Naive_Bayes              False        0.7429               0.6824            0.7066        0.6919
        Naive_Bayes               True        0.7470               0.7015            0.6827        0.6883
